# 03 — Feature Extraction (TF-IDF & Word Embeddings)

**Goal:** turn `description_processed` into numeric features two ways (TF-IDF, Word2Vec),
create a stratified train/test split (stratified on `category`, since `Research` is only ~5%),
and save everything so the modeling notebooks can load identical inputs.

**Structure:**
1. TF-IDF feature extraction (unigram vs unigram+bigram experiments)
2. Word2Vec training + averaged document vectors (+ optional BERT comparison)
3. Stratified train/test split + save all artifacts + comparison note


## TF-IDF Feature Extraction

Load the cleaned data, then experiment with a few TF-IDF configurations:
- unigrams only vs. unigrams + bigrams
- different `min_df` thresholds
- different `max_features` caps

We keep every fitted vectorizer + matrix in a dict so we can compare vocabulary size,
sparsity, and pick a final configuration before moving on.


In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_PATH = "../data/processed/tasks_nlp.csv"
OUT_DIR = "../models"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(8000, 14)


,task_id,task_description,description_processed,category,priority,status,created_date,due_date,days_to_deadline,estimated_hours,story_points,assignee_id,assignee_experience_years,assignee_open_tasks
0,TASK-12157,Add bulk export to CSV from the profile settings,add bulk export csv profile setting,Feature,Low,Blocked,2026-05-13,2026-05-31,18,9.9,2.0,USER-10,10.8,7
1,TASK-10453,Look into user reports about the mobile app,look user report mobile app,Research,High,Completed,2026-05-24,2026-06-18,25,12.9,2.0,USER-16,3.2,2
2,TASK-12716,Perform load testing on the checkout flow befo...,perform load testing checkout flow release,Testing,Low,In Progress,2026-04-09,2026-04-23,14,6.3,8.0,USER-11,0.8,18
3,TASK-16240,Fix the crash in the analytics pipeline when t...,fix crash analytics pipeline user submits empt...,Bug Fix,Medium,Completed,2026-02-14,2026-02-19,5,1.5,1.0,USER-05,2.8,3
4,TASK-15364,Build a filtering and sorting option for the a...,build filtering sorting option admin panel,Feature,Low,Completed,2026-03-16,2026-04-05,20,4.3,3.0,USER-18,8.1,15


In [2]:
# Sanity checks on the columns we depend on
assert "description_processed" in df.columns, "Missing description_processed column"
assert "category" in df.columns, "Missing category column"

# Drop any rows with empty/NaN processed text - TF-IDF/Word2Vec can't use them
before = len(df)
df = df.dropna(subset=["description_processed", "category"]).reset_index(drop=True)
df = df[df["description_processed"].str.strip() != ""].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with empty text/category")

print("\nClass distribution (%):")
print((df["category"].value_counts(normalize=True) * 100).round(2))

Dropped 0 rows with empty text/category

Class distribution (%):
category
Bug Fix          24.21
Feature          20.52
Testing          14.05
Maintenance      13.38
Documentation    10.74
Deployment       10.34
Research          6.76
Name: proportion, dtype: float64


In [3]:
# --- TF-IDF experiment grid ---
# We vary: ngram_range, min_df, max_features
configs = {
    "unigram_default":        dict(ngram_range=(1, 1), min_df=2,  max_features=5000),
    "unigram_bigram_default": dict(ngram_range=(1, 2), min_df=2,  max_features=5000),
    "unigram_bigram_strict":  dict(ngram_range=(1, 2), min_df=5,  max_features=10000),
    "unigram_bigram_wide":    dict(ngram_range=(1, 2), min_df=1,  max_features=None),
}

tfidf_results = {}

for name, params in configs.items():
    t0 = time.time()
    vec = TfidfVectorizer(**params, sublinear_tf=True)
    X = vec.fit_transform(df["description_processed"])
    elapsed = time.time() - t0

    tfidf_results[name] = {
        "vectorizer": vec,
        "matrix": X,
        "vocab_size": len(vec.vocabulary_),
        "shape": X.shape,
        "density_pct": round(100 * X.nnz / (X.shape[0] * X.shape[1]), 4),
        "fit_time_sec": round(elapsed, 2),
        "params": params,
    }

summary = pd.DataFrame({
    name: {
        "vocab_size": r["vocab_size"],
        "matrix_shape": r["shape"],
        "density_pct": r["density_pct"],
        "fit_time_sec": r["fit_time_sec"],
    }
    for name, r in tfidf_results.items()
}).T
summary

,vocab_size,matrix_shape,density_pct,fit_time_sec
unigram_default,220,"(8000, 220)",2.8567,0.05
unigram_bigram_default,1814,"(8000, 1814)",0.6386,0.07
unigram_bigram_strict,1459,"(8000, 1459)",0.7842,0.08
unigram_bigram_wide,1879,"(8000, 1879)",0.617,0.07


In [4]:
# Pick the configuration to carry forward as the "final" TF-IDF feature set.
# unigram_bigram_default is usually a good balance of vocab size vs signal -
# adjust this choice after reviewing the summary table above.
FINAL_TFIDF_CONFIG = "unigram_bigram_default"

final_tfidf_vectorizer = tfidf_results[FINAL_TFIDF_CONFIG]["vectorizer"]
final_tfidf_matrix = tfidf_results[FINAL_TFIDF_CONFIG]["matrix"]

print(f"Using config: {FINAL_TFIDF_CONFIG}")
print(f"Final TF-IDF matrix shape: {final_tfidf_matrix.shape}")

# Save the chosen vectorizer + matrix
joblib.dump(final_tfidf_vectorizer, f"{OUT_DIR}/tfidf_vectorizer.joblib")
joblib.dump(final_tfidf_matrix, f"{OUT_DIR}/tfidf_matrix.joblib")

# Also save the full experiment summary for the notebook's write-up
summary.to_csv(f"{OUT_DIR}/tfidf_experiment_summary.csv")
print("Saved tfidf_vectorizer.joblib, tfidf_matrix.joblib, tfidf_experiment_summary.csv")

Using config: unigram_bigram_default
Final TF-IDF matrix shape: (8000, 1814)
Saved tfidf_vectorizer.joblib, tfidf_matrix.joblib, tfidf_experiment_summary.csv


**message suggestion:**
`feat(features): add TF-IDF extraction with ngram/min_df/max_features experiments`


## Word2Vec Embeddings (+ optional BERT comparison)

Train a Word2Vec model directly on our tokenized task descriptions, then represent each
document as the **average** of its tokens' vectors. This gives a dense, low-dimensional
alternative to the sparse TF-IDF matrix.

Optionally, we also compute pretrained sentence-BERT embeddings for a quick side-by-side
comparison (skip this cell if `sentence-transformers` isn't installed / no internet access).


In [5]:
from gensim.models import Word2Vec

# Word2Vec expects a list of tokenized sentences (list of lists of tokens).
# description_processed is assumed to already be cleaned/lowercased text from the
# preprocessing notebook; we just whitespace-tokenize it here.
tokenized_docs = df["description_processed"].apply(lambda x: x.split()).tolist()

W2V_PARAMS = dict(
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,          # skip-gram (better for smaller datasets than CBOW)
    epochs=20,
    seed=RANDOM_STATE,
)

w2v_model = Word2Vec(sentences=tokenized_docs, **W2V_PARAMS)
print(f"Vocabulary size: {len(w2v_model.wv)}")
print(f"Vector size: {w2v_model.wv.vector_size}")

Vocabulary size: 220
Vector size: 100


In [6]:
def document_vector(tokens, model):
    """Average the Word2Vec vectors of all in-vocabulary tokens in a document.
    Returns a zero vector if none of the tokens are in the vocabulary."""
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if len(vecs) == 0:
        return np.zeros(model.wv.vector_size)
    return np.mean(vecs, axis=0)

w2v_doc_vectors = np.vstack([
    document_vector(tokens, w2v_model) for tokens in tokenized_docs
])
print(f"Averaged Word2Vec doc-vector matrix shape: {w2v_doc_vectors.shape}")

# Quick check: how many documents ended up as all-zero (no vocab overlap)?
zero_rows = np.sum(np.all(w2v_doc_vectors == 0, axis=1))
print(f"Documents with no in-vocabulary tokens: {zero_rows} / {len(w2v_doc_vectors)}")

Averaged Word2Vec doc-vector matrix shape: (8000, 100)
Documents with no in-vocabulary tokens: 0 / 8000


In [7]:
# Save Word2Vec model + averaged document vectors
w2v_model.save(f"{OUT_DIR}/word2vec_model.model")
joblib.dump(w2v_doc_vectors, f"{OUT_DIR}/word2vec_doc_vectors.joblib")
print("Saved word2vec_model.model, word2vec_doc_vectors.joblib")

Saved word2vec_model.model, word2vec_doc_vectors.joblib


### Optional: pretrained BERT sentence embeddings (for comparison only)

This cell requires `sentence-transformers` and internet access to download the pretrained
model. Skip / comment out if not available in your environment — it's optional and only used
for the qualitative comparison note below, not for the main deliverable.


In [8]:
USE_BERT_COMPARISON = False  # flip to True if sentence-transformers is available

if USE_BERT_COMPARISON:
    from sentence_transformers import SentenceTransformer

    bert_model = SentenceTransformer("all-MiniLM-L6-v2")
    bert_doc_vectors = bert_model.encode(
        df["description_processed"].tolist(),
        show_progress_bar=True,
        batch_size=64,
    )
    print(f"BERT doc-vector matrix shape: {bert_doc_vectors.shape}")
    joblib.dump(bert_doc_vectors, f"{OUT_DIR}/bert_doc_vectors.joblib")
    print("Saved bert_doc_vectors.joblib")
else:
    print("Skipping BERT comparison (USE_BERT_COMPARISON=False)")

Skipping BERT comparison (USE_BERT_COMPARISON=False)


**message suggestion:**
`feat(features): train Word2Vec model and build averaged doc vectors (+ optional BERT)`


## Stratified Train/Test Split & Saved Artifacts

`Research` is only ~5% of `category`, so a plain random split risks under-representing it
in the test set (or even in train, for a small enough sample). We use
`train_test_split(..., stratify=df["category"])` so both modelers train/evaluate on the
exact same rows.

We save:
- row indices for train/test (so *any* feature matrix can be sliced consistently)
- the label arrays
- a combined `feature_manifest.json` describing what was saved and where


In [9]:
TEST_SIZE = 0.2

train_idx, test_idx = train_test_split(
    df.index,
    test_size=TEST_SIZE,
    stratify=df["category"],
    random_state=RANDOM_STATE,
)

y_train = df.loc[train_idx, "category"]
y_test = df.loc[test_idx, "category"]

print(f"Train size: {len(train_idx)}  Test size: {len(test_idx)}")
print("\nTrain category distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))
print("\nTest category distribution (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

Train size: 6400  Test size: 1600

Train category distribution (%):


category
Bug Fix          24.22
Feature          20.53
Testing          14.05
Maintenance      13.38
Documentation    10.73
Deployment       10.33
Research          6.77
Name: proportion, dtype: float64

Test category distribution (%):
category
Bug Fix          24.19
Feature          20.50
Testing          14.06
Maintenance      13.38
Documentation    10.75
Deployment       10.38
Research          6.75
Name: proportion, dtype: float64


In [10]:
# Save split indices + labels so both modelers can reproduce the exact same split
joblib.dump(train_idx.to_numpy(), f"{OUT_DIR}/train_idx.joblib")
joblib.dump(test_idx.to_numpy(), f"{OUT_DIR}/test_idx.joblib")
joblib.dump(y_train, f"{OUT_DIR}/y_train.joblib")
joblib.dump(y_test, f"{OUT_DIR}/y_test.joblib")

# Also slice + save each feature set's train/test rows directly, for convenience
tfidf_train = final_tfidf_matrix[train_idx.to_numpy()]
tfidf_test = final_tfidf_matrix[test_idx.to_numpy()]
joblib.dump(tfidf_train, f"{OUT_DIR}/tfidf_train.joblib")
joblib.dump(tfidf_test, f"{OUT_DIR}/tfidf_test.joblib")

w2v_train = w2v_doc_vectors[train_idx.to_numpy()]
w2v_test = w2v_doc_vectors[test_idx.to_numpy()]
joblib.dump(w2v_train, f"{OUT_DIR}/w2v_train.joblib")
joblib.dump(w2v_test, f"{OUT_DIR}/w2v_test.joblib")

print("Saved train/test indices, labels, and pre-sliced TF-IDF / Word2Vec feature sets")

Saved train/test indices, labels, and pre-sliced TF-IDF / Word2Vec feature sets


In [11]:
import json

manifest = {
    "source_file": DATA_PATH,
    "n_rows_used": int(len(df)),
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "stratify_column": "category",
    "tfidf": {
        "config_used": FINAL_TFIDF_CONFIG,
        "params": tfidf_results[FINAL_TFIDF_CONFIG]["params"],
        "vocab_size": tfidf_results[FINAL_TFIDF_CONFIG]["vocab_size"],
        "files": ["tfidf_vectorizer.joblib", "tfidf_matrix.joblib",
                  "tfidf_train.joblib", "tfidf_test.joblib"],
    },
    "word2vec": {
        "params": W2V_PARAMS,
        "vocab_size": len(w2v_model.wv),
        "files": ["word2vec_model.model", "word2vec_doc_vectors.joblib",
                  "w2v_train.joblib", "w2v_test.joblib"],
    },
    "split_files": ["train_idx.joblib", "test_idx.joblib",
                    "y_train.joblib", "y_test.joblib"],
}

with open(f"{OUT_DIR}/feature_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(json.dumps(manifest, indent=2, default=str))

{
  "source_file": "../data/processed/tasks_nlp.csv",
  "n_rows_used": 8000,
  "test_size": 0.2,
  "random_state": 42,
  "stratify_column": "category",
  "tfidf": {
    "config_used": "unigram_bigram_default",
    "params": {
      "ngram_range": [
        1,
        2
      ],
      "min_df": 2,
      "max_features": 5000
    },
    "vocab_size": 1814,
    "files": [
      "tfidf_vectorizer.joblib",
      "tfidf_matrix.joblib",
      "tfidf_train.joblib",
      "tfidf_test.joblib"
    ]
  },
  "word2vec": {
    "params": {
      "vector_size": 100,
      "window": 5,
      "min_count": 2,
      "workers": 4,
      "sg": 1,
      "epochs": 20,
      "seed": 42
    },
    "vocab_size": 220,
    "files": [
      "word2vec_model.model",
      "word2vec_doc_vectors.joblib",
      "w2v_train.joblib",
      "w2v_test.joblib"
    ]
  },
  "split_files": [
    "train_idx.joblib",
    "test_idx.joblib",
    "y_train.joblib",
    "y_test.joblib"
  ]
}


### Comparison note: TF-IDF vs. Word2Vec (averaged) vs. BERT

| | TF-IDF | Word2Vec (avg) | BERT (optional) |
|---|---|---|---|
| **Dimensionality** | High, sparse (~5–10k, mostly zeros) | Low, dense (100) | Low, dense (384 for MiniLM) |
| **What it captures** | Exact word/phrase overlap, weighted by rarity (IDF) | Local co-occurrence patterns learned from *this* corpus only | Deep contextual/semantic meaning from a large pretrained corpus |
| **Handles synonyms / paraphrase** | No — different wording = different features | Somewhat, if the corpus is large enough to learn similar contexts | Yes, best of the three |
| **Data hunger** | Works fine on small/medium datasets | Needs a reasonably large corpus to learn good vectors; on a small task-description dataset the vectors may be noisy | Needs no training data (uses pretrained weights); best for small datasets |
| **Speed / cost to produce** | Fast, cheap, no external downloads | Fast to train locally, no internet needed | Requires downloading a pretrained model (needs internet) |
| **Interpretability** | High — each dimension = one word/bigram | Low — dimensions aren't individually interpretable | Low |
| **Best fit here** | Strong baseline, especially since categories likely have distinctive vocabulary | Reasonable second option, but our corpus is small so vectors may be less reliable than TF-IDF | Best semantic option if the extra dependency/download is acceptable |

**Recommendation:** start modeling with **TF-IDF** as the primary feature set (interpretable, distinctive vocabulary usually separates categories like `Research` well) and use **Word2Vec** averaged vectors as the alternative for comparison, per the assignment. Only add the BERT vectors if compute/time and network access allow — treat it as a stretch comparison rather than a required deliverable.

**message suggestion:**
`feat(features): add stratified train/test split, save feature matrices, add comparison note`
